In [4]:
# ===== Rhode – Recommendation system (KNN) =====
import pandas as pd
import requests
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.neighbors import NearestNeighbors

API_URL = "http://localhost:3000/api/products"
N_RECOMMENDATIONS = 3   # how many products to recommend for each product
PRICE_WEIGHT = 0.3      # how much the price counts compared to the text

# 1. Load the products from your API (Docker must be running)
products = pd.DataFrame(requests.get(API_URL).json())
products["price"] = products["price"].astype(float)
print(len(products), "products loaded")

# 2. Turn the text into numbers with TF-IDF (the category counts twice)
def product_text(row):
    parts = [row["name"], row["category"], row["category"],
             row.get("label") or "", row.get("tagline") or "", row.get("description") or ""]
    return " ".join(str(p) for p in parts).lower()

products["text"] = products.apply(product_text, axis=1)
text_features = normalize(TfidfVectorizer(stop_words="english").fit_transform(products["text"]))

# 3. Add the price, scaled between 0 and 1
price_features = MinMaxScaler().fit_transform(products[["price"]]) * PRICE_WEIGHT
features = hstack([text_features, csr_matrix(price_features)]).tocsr()

# 4. KNN: find the nearest neighbors (the closest one is the product itself, so we skip it)
knn = NearestNeighbors(n_neighbors=min(N_RECOMMENDATIONS + 1, len(products)), metric="euclidean")
knn.fit(features)
distances, indices = knn.kneighbors(features)

rows = []
for i, product_id in enumerate(products["id"]):
    rank = 1
    for dist, j in zip(distances[i], indices[i]):
        if j == i:
            continue
        rows.append({"product_id": int(product_id), "recommended_id": int(products["id"][j]),
                     "score": round(1 / (1 + float(dist)), 4), "rank_pos": rank})
        rank += 1
        if rank > N_RECOMMENDATIONS:
            break

# 5. Show the results with names, to check they make sense
names = dict(zip(products["id"], products["name"]))
results = pd.DataFrame(rows)
results["product"] = results["product_id"].map(names)
results["recommended"] = results["recommended_id"].map(names)

# 6. Create the SQL file to import into MySQL
lines = ["DELETE FROM product_recommendations;"]
for r in rows:
    lines.append("INSERT INTO product_recommendations (product_id, recommended_id, score, rank_pos) "
                 f"VALUES ({r['product_id']}, {r['recommended_id']}, {r['score']}, {r['rank_pos']});")
with open("recommendations.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print(f"recommendations.sql created with {len(rows)} recommendations ✅")

results[["product", "rank_pos", "recommended", "score"]]



9 products loaded
recommendations.sql created with 27 recommendations ✅


,product,rank_pos,recommended,score
0,peptide lip tint,1,spotwear,0.4140
1,peptide lip tint,2,pocket pearl,0.4139
2,peptide lip tint,3,rhode mirror,0.4139
3,pocket pearl,1,highlight milk,0.4478
4,pocket pearl,2,pocket blush,0.4372
5,pocket pearl,3,pocket bronzer,0.4339
6,pocket bronzer,1,pocket pearl,0.4339
7,pocket bronzer,2,pocket blush,0.4330
8,pocket bronzer,3,highlight milk,0.4281
9,highlight milk,1,pocket pearl,0.4478
